## Grouped Subject-Aware Modeling and Uncertainty

This notebook reports final grouped evaluation results using the preferred compact setup selected in notebook 02.

Scope remains compact and interview-defensible:
1. grouped regression forecasting,
2. grouped current-activity classification,
3. grouped split conformal uncertainty for regression.

This notebook also surfaces focused ablation findings and adds operational diagnostics:
- global versus activity-conditioned conformal comparison,
- coverage and interval-width breakdowns by activity and subject,
- residual and interval-failure diagnostics,
- classification confidence calibration and abstention tradeoffs,
- explicit operating-envelope and limitations framing.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from pamap2_telemetry.evaluate import ALPHA, RANDOM_SEED, run_grouped_evaluation

REGRESSION_PROCESSED_PATH = REPO_ROOT / "data" / "processed" / "pamap2_model_table_regression.parquet"
CLASSIFICATION_PROCESSED_PATH = REPO_ROOT / "data" / "processed" / "pamap2_model_table_classification.parquet"
METRICS_DIR = REPO_ROOT / "artifacts" / "metrics"
FIGURES_DIR = REPO_ROOT / "artifacts" / "figures"
MODELS_DIR = REPO_ROOT / "artifacts" / "models"
PREFERRED_SETUP_PATH = METRICS_DIR / "grouped_cv_preferred_setup_summary.csv"

preferred_setup_df = pd.read_csv(PREFERRED_SETUP_PATH)
preferred_target_col = str(preferred_setup_df.iloc[0]["preferred_target_col"])

print(f"Repo root: {REPO_ROOT}")
print(f"Regression-ready table exists: {REGRESSION_PROCESSED_PATH.exists()}")
print(f"Classification-ready table exists: {CLASSIFICATION_PROCESSED_PATH.exists()}")
print(f"Preferred regression target: {preferred_target_col}")
print(f"Using random seed: {RANDOM_SEED}")
print(f"Conformal alpha: {ALPHA}")


In [ ]:
if not REGRESSION_PROCESSED_PATH.exists():
    raise FileNotFoundError(f"Missing regression processed model table: {REGRESSION_PROCESSED_PATH}")
if not CLASSIFICATION_PROCESSED_PATH.exists():
    raise FileNotFoundError(
        f"Missing classification processed model table: {CLASSIFICATION_PROCESSED_PATH}"
    )

reg_model_df = pd.read_parquet(REGRESSION_PROCESSED_PATH).copy()
cls_model_df = pd.read_parquet(CLASSIFICATION_PROCESSED_PATH).copy()

reg_model_df = reg_model_df.sort_values(["subject_id", "timestamp_s"]).reset_index(drop=True)
cls_model_df = cls_model_df.sort_values(["subject_id", "timestamp_s"]).reset_index(drop=True)

reg_required_columns = [
    "subject_id",
    "timestamp_s",
    "activity_target",
    "activity_label",
    "heart_rate_bpm",
    preferred_target_col,
]
cls_required_columns = [
    "subject_id",
    "timestamp_s",
    "activity_target",
    "activity_label",
    "heart_rate_bpm",
]

missing_reg_required = [column for column in reg_required_columns if column not in reg_model_df.columns]
missing_cls_required = [column for column in cls_required_columns if column not in cls_model_df.columns]
if missing_reg_required:
    raise ValueError(f"Regression table is missing required columns: {missing_reg_required}")
if missing_cls_required:
    raise ValueError(f"Classification table is missing required columns: {missing_cls_required}")

reg_duplicate_count = int(reg_model_df.duplicated(subset=["subject_id", "timestamp_s"]).sum())
cls_duplicate_count = int(cls_model_df.duplicated(subset=["subject_id", "timestamp_s"]).sum())
if reg_duplicate_count > 0:
    raise ValueError(f"Found duplicate subject-second rows in regression table: {reg_duplicate_count}")
if cls_duplicate_count > 0:
    raise ValueError(f"Found duplicate subject-second rows in classification table: {cls_duplicate_count}")

print(f"Regression rows: {len(reg_model_df):,}")
print(f"Classification rows: {len(cls_model_df):,}")
print(f"Classification row gain vs regression: {len(cls_model_df) - len(reg_model_df):,}")
print(f"Columns (regression table): {len(reg_model_df.columns)}")
print(f"Columns (classification table): {len(cls_model_df.columns)}")
print(f"Regression subjects: {sorted(reg_model_df['subject_id'].unique().tolist())}")
print(f"Classification subjects: {sorted(cls_model_df['subject_id'].unique().tolist())}")
print(f"Target columns: {[c for c in reg_model_df.columns if c.startswith('hr_target_')]}")
display(reg_model_df.head())


## Run Grouped LOSO Evaluation

This cell executes the full grouped workflow and writes all metric tables and figures used for model selection and error-breakdown reporting.


In [ ]:
results = run_grouped_evaluation(
    regression_processed_path=REGRESSION_PROCESSED_PATH,
    classification_processed_path=CLASSIFICATION_PROCESSED_PATH,
    metrics_dir=METRICS_DIR,
    figures_dir=FIGURES_DIR,
    models_dir=MODELS_DIR,
    random_seed=RANDOM_SEED,
    alpha=ALPHA,
    regression_target_col=preferred_target_col,
)

print("Grouped evaluation complete for preferred setup.")
print("Saved grouped metrics and figures with grouped_cv_ file names.")
print("Returned result keys:", sorted(results.keys()))


In [ ]:
feature_ablation_df = pd.read_csv(METRICS_DIR / "grouped_cv_feature_ablation_summary.csv")
target_comparison_df = pd.read_csv(METRICS_DIR / "grouped_cv_target_comparison_summary.csv")
fill_sensitivity_df = pd.read_csv(METRICS_DIR / "grouped_cv_fill_sensitivity_summary.csv")

print("Feature ablation summary:")
display(feature_ablation_df.sort_values("best_mean_mae").reset_index(drop=True))

print("Target comparison summary:")
display(target_comparison_df.sort_values("best_mean_mae").reset_index(drop=True))

print("Fill sensitivity summary:")
display(fill_sensitivity_df.sort_values("best_mean_mae").reset_index(drop=True))


In [ ]:
regression_fold_df = results["regression_fold"].copy()
regression_summary_df = results["regression_summary"].copy()
classification_fold_df = results["classification_fold"].copy()
classification_summary_df = results["classification_summary"].copy()
classification_per_class_df = results["classification_per_class"].copy()

print("Regression fold-level metrics:")
display(regression_fold_df.sort_values(["model", "fold"]).reset_index(drop=True))

print("Regression grouped CV summary (mean/std/min/max across folds):")
display(regression_summary_df.sort_values("rank"))

print("Classification fold-level metrics:")
display(classification_fold_df.sort_values(["model", "fold"]).reset_index(drop=True))

print("Classification grouped CV summary (mean/std/min/max across folds):")
display(classification_summary_df.sort_values("rank"))

print("Selected classification model per-class performance:")
display(classification_per_class_df.sort_values("support", ascending=False).reset_index(drop=True))


## Final Model Selection and Breakdown Reporting

Selection is driven by grouped CV summaries only, not one validation subject.

Rules used:
- Regression: lowest mean MAE across LOSO folds, tie-break by MAE standard deviation then mean RMSE.
- Classification: highest mean macro F1 across LOSO folds, tie-break by macro F1 standard deviation then mean accuracy.


In [ ]:
selected_models_df = results["selected_models"].copy()

regression_by_subject_df = results["regression_by_subject"].copy()
regression_by_activity_df = results["regression_by_activity"].copy()
classification_by_subject_df = results["classification_by_subject"].copy()
classification_by_activity_df = results["classification_by_activity"].copy()

print("Selected-model summary table:")
display(selected_models_df)

print("Regression performance by subject:")
display(regression_by_subject_df.sort_values("subject_id").reset_index(drop=True))

print("Regression performance by activity:")
display(regression_by_activity_df.sort_values("rows", ascending=False).reset_index(drop=True))

print("Classification performance by subject:")
display(classification_by_subject_df.sort_values("subject_id").reset_index(drop=True))

print("Classification performance by activity:")
display(classification_by_activity_df.sort_values("rows", ascending=False).reset_index(drop=True))


## Leakage and Coverage Checks

This section confirms fold-wise subject isolation and summarizes grouped conformal interval quality.


In [ ]:
reg_subject_count = int(reg_model_df["subject_id"].nunique())
cls_subject_count = int(cls_model_df["subject_id"].nunique())

reg_subject_coverage = (
    regression_fold_df.groupby(["model", "test_subject_id"], as_index=False)
    .size()
    .rename(columns={"size": "row_count"})
)
cls_subject_coverage = (
    classification_fold_df.groupby(["model", "test_subject_id"], as_index=False)
    .size()
    .rename(columns={"size": "row_count"})
)

reg_subjects_per_model = regression_fold_df.groupby("model")["test_subject_id"].nunique()
cls_subjects_per_model = classification_fold_df.groupby("model")["test_subject_id"].nunique()

if not (reg_subjects_per_model == reg_subject_count).all():
    raise ValueError("Regression grouped CV did not cover each subject exactly once per model.")
if not (cls_subjects_per_model == cls_subject_count).all():
    raise ValueError("Classification grouped CV did not cover each subject exactly once per model.")

print("Leakage guard checks passed.")
print(f"Unique subjects in regression table: {reg_subject_count}")
print(f"Unique subjects in classification table: {cls_subject_count}")
print("Regression subjects per model:")
display(reg_subjects_per_model.rename("unique_test_subjects"))
print("Classification subjects per model:")
display(cls_subjects_per_model.rename("unique_test_subjects"))


## Grouped Conformal Uncertainty and Diagnostics

Conformal intervals are computed fold by fold with disjoint subject sets:
- proper-train subjects fit the selected regression model,
- one separate calibration subject sets conformal margins,
- the held-out test subject receives interval predictions.

This section compares two lightweight variants:
- global split conformal (single margin per fold),
- activity-conditioned split conformal (activity-level margins with fallback to global when calibration support is small).

It then reports diagnostics for the selected variant.


In [ ]:
conformal_fold_df = results["conformal_fold"].copy()
conformal_summary_df = results["conformal_summary"].copy()
conformal_summary_all_df = results["conformal_summary_all_variants"].copy()
conformal_variant_comparison_df = results["conformal_variant_comparison"].copy()
conformal_by_subject_df = results["conformal_by_subject"].copy()
conformal_by_activity_df = results["conformal_by_activity"].copy()
preferred_conformal_variant = results["preferred_conformal_variant"]

print(f"Selected conformal variant: {preferred_conformal_variant}")
print("Conformal fold summary for selected variant:")
display(conformal_fold_df.sort_values("fold").reset_index(drop=True))

print("Conformal variant comparison (coverage/width tradeoff):")
display(conformal_variant_comparison_df)

print("Conformal aggregate summary by variant:")
display(conformal_summary_all_df.sort_values("calibration_variant").reset_index(drop=True))

print("Conformal coverage by subject (selected variant):")
display(conformal_by_subject_df.sort_values("subject_id").reset_index(drop=True))

print("Conformal coverage by activity (selected variant):")
display(conformal_by_activity_df.sort_values("rows", ascending=False).reset_index(drop=True))


## Residual, Failure, and Confidence Diagnostics

This section uses uncertainty outputs as diagnostics instead of decorative intervals.

Reported here:
- where large errors and interval failures cluster,
- residual summary statistics for the selected regression model,
- classification probability calibration and reliability bins,
- abstention-style tradeoff when low-confidence predictions are withheld.


In [ ]:
uncertainty_failure_by_activity_df = results["uncertainty_failure_by_activity"].copy()
uncertainty_failure_by_subject_df = results["uncertainty_failure_by_subject"].copy()
regression_residual_summary_df = results["regression_residual_summary"].copy()
uncertainty_operating_envelope_df = results["uncertainty_operating_envelope"].copy()

classification_calibration_summary_df = results["classification_calibration_summary"].copy()
classification_reliability_df = results["classification_reliability"].copy()
classification_abstention_df = results["classification_abstention"].copy()

print("Uncertainty failure hotspots by activity:")
display(
    uncertainty_failure_by_activity_df.sort_values(
        ["interval_failure_rate", "mean_abs_error"],
        ascending=[False, False],
    ).head(10).reset_index(drop=True)
)

print("Uncertainty failure hotspots by subject:")
display(
    uncertainty_failure_by_subject_df.sort_values(
        ["interval_failure_rate", "mean_abs_error"],
        ascending=[False, False],
    ).reset_index(drop=True)
)

print("Residual summary for selected regression model:")
display(regression_residual_summary_df)

print("Operating envelope by activity:")
display(
    uncertainty_operating_envelope_df[[
        "activity_label",
        "rows",
        "empirical_coverage",
        "coverage_gap_vs_target",
        "mean_abs_error",
        "mean_interval_width",
        "envelope_status",
    ]].sort_values(["envelope_status", "rows"], ascending=[True, False]).reset_index(drop=True)
)

print("Classification calibration summary:")
display(classification_calibration_summary_df)

print("Classification reliability bins:")
display(classification_reliability_df.reset_index(drop=True))

print("Low-confidence abstention analysis:")
display(classification_abstention_df.reset_index(drop=True))


In [ ]:
grouped_metric_files = sorted(METRICS_DIR.glob("grouped_cv_*.csv"))
grouped_figure_files = sorted(FIGURES_DIR.glob("grouped_cv_*.png"))

print("Grouped metric artifacts:")
for path in grouped_metric_files:
    print(path.relative_to(REPO_ROOT))

print("\nGrouped figure artifacts:")
for path in grouped_figure_files:
    print(path.relative_to(REPO_ROOT))


## Limits and Operating Envelope

What this workflow demonstrates:
- subject-held-out activity classification is viable with compact telemetry features,
- short-horizon heart-rate forecasting improves over persistence but still depends partly on recent HR state,
- conformal intervals can be calibrated globally and stress-tested by activity and subject.

What this workflow does not demonstrate:
- deployment robustness under sensor drift, missing channels, or streaming latency,
- calibrated confidence transfer to unseen devices or populations beyond PAMAP2 subjects,
- long-horizon physiological forecasting beyond short telemetry windows.

Current weak points to communicate clearly:
- specific activities and subjects with under-target coverage or higher interval failures,
- lower-confidence classification regimes where abstention may be preferred,
- residual tails where large errors still occur even with calibrated intervals.

Production-minded next improvements (still lightweight):
- collect richer calibration sets for weak activities before tightening interval policy,
- add a simple confidence gate policy for classification decisions,
- add targeted features or class grouping for activities with persistent failure clusters.


In [ ]:
# Compact interview-oriented summary pulled from grouped evaluation and ablation outputs.
reg_choice = selected_models_df[selected_models_df["task"] == "regression"].iloc[0]
cls_choice = selected_models_df[selected_models_df["task"] == "classification"].iloc[0]
preferred_setup_df = pd.read_csv(METRICS_DIR / "grouped_cv_preferred_setup_summary.csv")

best_reg_row = regression_summary_df.sort_values("mean_mae").iloc[0]
persistence_reg_row = regression_summary_df[regression_summary_df["model"] == "persistence_current_hr"].iloc[0]
persistence_gap = persistence_reg_row["mean_mae"] - best_reg_row["mean_mae"]

feature_baseline_row = feature_ablation_df[feature_ablation_df["feature_set"] == "baseline"].iloc[0]
feature_upgraded_row = feature_ablation_df[feature_ablation_df["feature_set"] == "upgraded"].iloc[0]
feature_gain = feature_baseline_row["best_mean_mae"] - feature_upgraded_row["best_mean_mae"]

target_direct_row = target_comparison_df[target_comparison_df["target_variant"] == "direct_30s"].iloc[0]
target_best_row = target_comparison_df.sort_values("best_mean_mae").iloc[0]
target_gain = target_direct_row["best_mean_mae"] - target_best_row["best_mean_mae"]

conformal_variant_comparison_df = results["conformal_variant_comparison"].copy()
uncertainty_failure_by_activity_df = results["uncertainty_failure_by_activity"].copy()
classification_calibration_summary_df = results["classification_calibration_summary"].copy()
classification_abstention_df = results["classification_abstention"].copy()

calibration_row = classification_calibration_summary_df.iloc[0]
abstain_row = classification_abstention_df.iloc[
    (classification_abstention_df["confidence_threshold"] - 0.8).abs().argmin()
]

print("Preferred setup:")
display(preferred_setup_df)

print("Final model choices from grouped CV:")
print(
    f"Regression: {reg_choice['selected_model']} | mean MAE={reg_choice['selected_mean_mae']:.3f} "
    f"(runner-up margin={reg_choice['winner_margin']:.3f})"
)
print(
    f"Classification: {cls_choice['selected_model']} | mean macro F1={cls_choice['selected_mean_macro_f1']:.3f} "
    f"(runner-up margin={cls_choice['winner_margin']:.3f})"
)

print("\nFeature/target impact on downstream failure risk:")
print(f"Feature upgrade MAE gain vs baseline: {feature_gain:.3f} bpm")
print(f"Target reformulation MAE gain vs direct t+30s: {target_gain:.3f} bpm")
print(
    "Interpretation: lower base error from compact feature/target improvements reduces broad failure pressure, "
    "but interval failures remain concentrated in specific activities."
)

print("\nPersistence sensitivity check:")
print(f"Best regression model MAE: {best_reg_row['mean_mae']:.3f}")
print(f"Persistence baseline MAE: {persistence_reg_row['mean_mae']:.3f}")
print(f"MAE gain vs persistence: {persistence_gap:.3f}")
if persistence_gap < 0.25:
    print("Interpretation: forecast task looks strongly persistence-driven.")
elif persistence_gap < 0.75:
    print("Interpretation: forecast task is partly persistence-driven, but model features add clear value.")
else:
    print("Interpretation: upgraded setup captures meaningful signal beyond persistence.")

print("\nConformal variant decision:")
display(conformal_variant_comparison_df)

print("\nHighest uncertainty-failure activities:")
display(
    uncertainty_failure_by_activity_df.sort_values("interval_failure_rate", ascending=False).head(5)[
        [
            "activity_label",
            "rows",
            "empirical_coverage",
            "interval_failure_rate",
            "mean_abs_error",
            "mean_interval_width",
        ]
    ]
)

print("\nClassification confidence diagnostics:")
print(
    f"Brier={calibration_row['multiclass_brier_score']:.3f} | "
    f"ECE_10={calibration_row['ece_10']:.3f} | "
    f"overconfidence gap={calibration_row['overconfidence_gap']:.3f}"
)
print(
    f"At confidence >=0.8: retained fraction={abstain_row['retained_fraction']:.3f}, "
    f"retained accuracy={abstain_row['retained_accuracy']:.3f}"
)


## Completion Checklist

This notebook now reports the final preferred setup with compact, subject-aware evaluation:
- focused ablation outputs for feature, target, and fill-policy decisions,
- grouped LOSO model selection for regression and classification,
- global versus activity-conditioned conformal comparison,
- by-subject and by-activity uncertainty diagnostics with coverage and width,
- residual distribution and large-error/interval-failure hotspot analysis,
- classification calibration metrics and reliability bins,
- low-confidence abstention tradeoff table,
- explicit operating-envelope and limitations framing.

All non-obvious choices are documented in project docs and decision log.
